<a href="https://colab.research.google.com/github/Amazeble/chatterbox-finetuning/blob/main/chatterbox-finetuning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a># 🎙️ Chatterbox TTS Fine-Tuning on Google ColabThis notebook provides a complete workflow for fine-tuning Chatterbox TTS models (Standard and Turbo) with LoRA support.## Workflow Overview- **STEP 1**: Configuration - Set all training parameters and paths- **STEP 2**: Google Drive Connection & Copy Dataset- **STEP 3**: Clone Repository & Install Dependencies- **STEP 4**: Download & Prepare Models (setup.py)- **STEP 5**: Train Model (+ Auto Merge LoRA if enabled)- **OPTIONAL STEP 6**: Merge LoRA Adapter (Manual)- **OPTIONAL STEP 7**: Inference (Generate Speech)

## STEP 1: ConfigurationConfigure all training parameters, dataset paths, and Google Drive paths here.

In [ ]:
#@title ⚙️ STEP 1: Configure All Parameters#@markdown **📁 Google Drive Paths:**source_path = '/content/drive/MyDrive/MyTTSDataset/' #@param {type:"string"}dest_path = '/content/chatterbox-finetuning/' #@param {type:"string"}#@markdown **🎯 Training Options:**is_turbo = True #@param {type:"boolean"}is_lora = True #@param {type:"boolean"}is_merge_lora = True #@param {type:"boolean"}#@markdown **📊 Dataset Format:**ljspeech = True #@param {type:"boolean"}json_format = False #@param {type:"boolean"}preprocess = "auto" #@param ["True", "False", "auto"]#@markdown **🔧 Hyperparameters:**batch_size = 8 #@param {type:"slider", min:1, max:16, step:1}learning_rate = 0.0001 #@param {type:"number"}num_epochs = 10 #@param {type:"slider", min:1, max:50, step:1}new_vocab_size = 52260 #@param {type:"integer"}#@markdown **🏗️ Model Architecture:**lora_r = 128 #@param {type:"integer"}lora_alpha = 256 #@param {type:"integer"}#@markdown **📂 Output & Paths:**model_dir = "./pretrained_models" #@param {type:"string"}csv_path = "./MyTTSDataset/metadata.csv" #@param {type:"string"}wav_dir = "./MyTTSDataset/wavs" #@param {type:"string"}preprocessed_dir = "./MyTTSDataset/preprocess" #@param {type:"string"}output_dir = "./chatterbox_output" #@param {type:"string"}#@markdown **🗣️ Inference Settings:**inference_prompt_path = "./speaker_reference/2.wav" #@param {type:"string"}inference_test_text = "Merhaba, sesimi geliştirmem oldukça uzun zaman aldı ve şimdi sahip olduğuma göre, sessiz kalmayacağım." #@param {type:"string"}import osimport re# Detect environment and set working directoryif os.path.exists('/content'):    WORKSPACE = '/content'    %cd /content    print("Detected Google Colab environment")else:    WORKSPACE = '/workspace'    %cd "$WORKSPACE"    print("Running in local/other environment")# Clone the repository if it doesn't exist (needed to write config)if not os.path.exists('chatterbox-finetuning'):    print("Cloning Chatterbox Finetuning repository...")    !git clone https://github.com/Amazeble/chatterbox-finetuning.git    %cd chatterbox-finetuningelse:    %cd chatterbox-finetuning# Create src directory if it doesn't existos.makedirs('src', exist_ok=True)# Read current config file if it exists, otherwise create boilerplateconfig_template = """from dataclasses import dataclass, fieldfrom typing import List, Literal, Optionalimport osimport globdef should_run_preprocessing(config) -> bool:    if config.preprocess is True:        return True    elif config.preprocess is False:        return False    elif config.preprocess == "auto":        if not os.path.exists(config.preprocessed_dir):            return True        wav_files = glob.glob(os.path.join(config.wav_dir, "*.wav"))        wav_count = len(wav_files)        pt_files = glob.glob(os.path.join(config.preprocessed_dir, "*.pt"))        pt_count = len(pt_files)        if wav_count != pt_count:            return True        return False    return True@dataclassclass TrainConfig:    # --- Paths ---    model_dir: str = "./pretrained_models"    csv_path: str = "./MyTTSDataset/metadata.csv"    wav_dir: str = "./MyTTSDataset/wavs"    preprocessed_dir = "./MyTTSDataset/preprocess"    output_dir: str = "./chatterbox_output"        is_inference = False    inference_prompt_path: str = "./speaker_reference/2.wav"    inference_test_text: str = "Merhaba, sesimi geliştirmem oldukça uzun zaman aldı ve şimdi sahip olduğuma göre, sessiz kalmayacağım."    ljspeech = True    json_format = False    preprocess: Optional[Literal[True, False, "auto"]] = "auto"        is_turbo: bool = True    is_lora: bool = True    is_merge_lora: bool = True    lora_r: int = 128    lora_alpha: int = 256    turbo_lora_target_modules: List[str] = field(default_factory=lambda: ["c_attn", "c_proj", "c_fc", "spkr_enc"])    lora_target_modules: List[str] = field(default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "spkr_enc"])    lora_modules_to_save: List[str] = field(default_factory=lambda: ["text_emb", "text_head"])        new_vocab_size: int = 52260 if is_turbo else 2454     batch_size: int = 8    grad_accum: int = 4    learning_rate: float = 1e-4 if is_lora else 1e-5    num_epochs: int = 10 if is_lora else 30        save_steps: int = 500    save_total_limit: int = 5    dataloader_num_workers: int = 8    start_text_token = 255    stop_text_token = 0    max_text_len: int = 256    max_speech_len: int = 850    prompt_duration: float = 3.0"""if not os.path.exists('src/config.py'):    with open('src/config.py', 'w') as f:        f.write(config_template)with open('src/config.py', 'r') as f:    config_content = f.read()# Convert booleans to script stringsturbo_val = "True" if is_turbo else "False"lora_val = "True" if is_lora else "False"merge_lora_val = "True" if is_merge_lora else "False"ljspeech_val = "True" if ljspeech else "False"json_val = "True" if json_format else "False"# Update configurations using regexconfig_content = re.sub(r'is_turbo:\s*bool\s*=\s*.*', f'is_turbo: bool = {turbo_val}', config_content)config_content = re.sub(r'is_lora:\s*bool\s*=\s*.*', f'is_lora: bool = {lora_val}', config_content)config_content = re.sub(r'is_merge_lora:\s*bool\s*=\s*.*', f'is_merge_lora: bool = {merge_lora_val}', config_content)config_content = re.sub(r'ljspeech\s*=\s*.*', f'ljspeech = {ljspeech_val}', config_content)config_content = re.sub(r'json_format\s*=\s*.*', f'json_format = {json_val}', config_content)config_content = re.sub(r'preprocess:\s*Optional\[Literal\[True, False, "auto"\]\]\s*=\s*.*', f'preprocess: Optional[Literal[True, False, "auto"]] = "{preprocess}"', config_content)config_content = re.sub(r'batch_size:\s*int\s*=\s*.*', f'batch_size: int = {batch_size}', config_content)config_content = re.sub(r'learning_rate:\s*float\s*=\s*.*', f'learning_rate: float = {learning_rate}', config_content)config_content = re.sub(r'num_epochs:\s*int\s*=\s*.*', f'num_epochs: int = {num_epochs}', config_content)config_content = re.sub(r'new_vocab_size:\s*int\s*=\s*.*', f'new_vocab_size: int = {new_vocab_size}', config_content)config_content = re.sub(r'lora_r:\s*int\s*=\s*.*', f'lora_r: int = {lora_r}', config_content)config_content = re.sub(r'lora_alpha:\s*int\s*=\s*.*', f'lora_alpha: int = {lora_alpha}', config_content)config_content = re.sub(r'model_dir:\s*str\s*=\s*".*"', f'model_dir: str = "{model_dir}"', config_content)config_content = re.sub(r'csv_path:\s*str\s*=\s*".*"', f'csv_path: str = "{csv_path}"', config_content)config_content = re.sub(r'wav_dir:\s*str\s*=\s*".*"', f'wav_dir: str = "{wav_dir}"', config_content)config_content = re.sub(r'preprocessed_dir\s*=\s*".*"', f'preprocessed_dir = "{preprocessed_dir}"', config_content)config_content = re.sub(r'output_dir:\s*str\s*=\s*".*"', f'output_dir: str = "{output_dir}"', config_content)config_content = re.sub(r'inference_prompt_path:\s*str\s*=\s*".*"', f'inference_prompt_path: str = "{inference_prompt_path}"', config_content)config_content = re.sub(r'inference_test_text:\s*str\s*=\s*".*"', f'inference_test_text: str = "{inference_test_text}"', config_content)# Save the updated configurationswith open('src/config.py', 'w') as f:    f.write(config_content)# Save source_path and dest_path to a separate config file for drive operationsdrive_config = f"""# Google Drive ConfigurationSOURCE_PATH = '{source_path}'DEST_PATH = '{dest_path}'"""with open('src/drive_config.py', 'w') as f:    f.write(drive_config)# Show confirmationfrom IPython.display import clear_outputclear_output()print("="*60)print("✅ STEP 1: Configuration Applied Successfully!")print("="*60)print("\n📁 Google Drive Paths:")print(f"  - source_path: {source_path}")print(f"  - dest_path: {dest_path}")print("\n🎯 Training Options:")print(f"  - is_turbo: {turbo_val}")print(f"  - is_lora: {lora_val}")print(f"  - is_merge_lora: {merge_lora_val}")print("\n📊 Dataset Format:")print(f"  - ljspeech: {ljspeech_val}")print(f"  - json_format: {json_val}")print(f"  - preprocess: {preprocess}")print("\n🔧 Hyperparameters:")print(f"  - batch_size: {batch_size}")print(f"  - learning_rate: {learning_rate}")print(f"  - num_epochs: {num_epochs}")print(f"  - new_vocab_size: {new_vocab_size}")print(f"  - lora_r: {lora_r}")print(f"  - lora_alpha: {lora_alpha}")print("\n📂 Paths:")print(f"  - model_dir: {model_dir}")print(f"  - csv_path: {csv_path}")print(f"  - wav_dir: {wav_dir}")print(f"  - preprocessed_dir: {preprocessed_dir}")print(f"  - output_dir: {output_dir}")print("\n🗣️ Inference:")print(f"  - inference_prompt_path: {inference_prompt_path}")print(f"  - inference_test_text: {inference_test_text[:50]}...")print("\n" + "="*60)print("⚠️ IMPORTANT: Proceed to STEP 2 to connect Google Drive!")print("="*60)

## STEP 2: Google Drive Connection & Copy DatasetConnect to Google Drive and copy your dataset to the Colab environment.

In [ ]:
#@title 📁 STEP 2: Connect Google Drive & Copy Datasetimport osfrom google.colab import drive# Import paths from drive_config created in STEP 1try:    from src.drive_config import SOURCE_PATH, DEST_PATHexcept ImportError:    print("⚠️ drive_config.py not found! Please run STEP 1 first.")    SOURCE_PATH = '/content/drive/MyDrive/MyTTSDataset/'    DEST_PATH = '/content/chatterbox-finetuning/'    print(f"Using default paths:")    print(f"  SOURCE_PATH: {SOURCE_PATH}")    print(f"  DEST_PATH: {DEST_PATH}")# Mount Google Driveprint("Mounting Google Drive...")drive.mount('/content/drive')# Check if source path existsif os.path.exists(SOURCE_PATH):    print(f"\n✅ Source path found: {SOURCE_PATH}")    print("Copying dataset to working directory...")        # Create destination directory if it doesn't exist    os.makedirs(DEST_PATH, exist_ok=True)        # Copy files    !cp -r {SOURCE_PATH}/* {DEST_PATH}        print(f"✅ Dataset copied successfully to {DEST_PATH}")    print("\nVerifying copied files:")    !ls -la {DEST_PATH}else:    print(f"⚠️ Source path not found: {SOURCE_PATH}")    print("Please ensure your dataset is in the correct location in Google Drive.")

## STEP 3: Clone Repository & Install DependenciesClone the repository (if not already done) and install all required dependencies.

In [ ]:
#@title 🔧 STEP 3: Clone Repository & Install Dependenciesimport os# Ensure we're in the right directoryif os.path.exists('/content/chatterbox-finetuning'):    %cd /content/chatterbox-finetuningelif os.path.exists('/workspace/chatterbox-finetuning'):    %cd /workspace/chatterbox-finetuningelse:    print("Repository not found! Please run STEP 1 first.")# Install FFmpegprint("Installing FFmpeg...")!apt-get update && apt-get install -y ffmpeg# Install Python dependenciesprint("\nInstalling Python dependencies...")!pip install -r requirements.txtprint("\n✅ Dependencies installed successfully!")

## STEP 4: Download & Prepare Models (setup.py)Download the base models and prepare the environment.

In [ ]:
#@title \ud83d\udce5 STEP 4: Download & Prepare Models (setup.py)%cd /content/chatterbox-finetuningprint("Running setup.py to download models...")print("This may take several minutes depending on your internet connection.")print("\nNote: setup.py will read 'is_turbo' from src/config.py (configured in STEP 1)")# Verify config.py exists and show is_turbo settingif os.path.exists('src/config.py'):    with open('src/config.py', 'r') as f:        config_content = f.read()        import re        turbo_match = re.search(r'is_turbo:\s*bool\s*=\s*(True|False)', config_content)        if turbo_match:            print(f"\n\u2705 Config verified: is_turbo = {turbo_match.group(1)}")        else:            print("\n\u26a0\ufe0f Warning: Could not verify is_turbo setting in config.py")else:    print("\n\u26a0\ufe0f Warning: src/config.py not found! Please run STEP 1 first.")# Run setup.py (it will read is_turbo from src/config.py)!python setup.pyprint("\n\u2705 Setup completed!")print("Check the output above for the vocab_size if using Turbo mode.")

## STEP 5: Train ModelTrain the model using the configuration from STEP 1. If `is_merge_lora` is enabled, the LoRA adapter will be automatically merged after training.

In [ ]:
#@title 🏋️ STEP 5: Train Model (+ Auto Merge LoRA)%cd /content/chatterbox-finetuningprint("="*60)print("🏋️ STEP 5: Train Model")print("="*60)# Read config to check settingswith open('src/config.py', 'r') as f:    config_content = f.read()print("\n📋 Current Configuration:")print("-" * 60)import resettings = {    'Turbo Mode': r'is_turbo:\s*bool\s*=\s*(\w+)',    'LoRA Training': r'is_lora:\s*bool\s*=\s*(\w+)',    'Auto Merge LoRA': r'is_merge_lora:\s*bool\s*=\s*(\w+)',    'Batch Size': r'batch_size:\s*int\s*=\s*(\d+)',    'Learning Rate': r'learning_rate:\s*float\s*=\s*([\d.e+-]+)',    'Epochs': r'num_epochs:\s*int\s*=\s*(\d+)',    'Preprocess': r"preprocess:\s*Optional\[Literal\[True, False, \"auto\"\]\]\s*=\s*[\"']?([^\"'\s]+)[\"']?",    'LJSpeech Format': r'ljspeech\s*=\s*(\w+)',    'JSON Format': r'json_format\s*=\s*(\w+)'}for name, pattern in settings.items():    match = re.search(pattern, config_content)    if match:        print(f"{name}: {match.group(1)}")print("-" * 60)# Check if preprocessing should runpreprocess_val = Nonepreprocess_match = re.search(r"preprocess:\s*Optional\[Literal\[True, False, \"auto\"\]\]\s*=\s*[\"']?([^\"'\s]+)[\"']?", config_content)if preprocess_match:    preprocess_val = preprocess_match.group(1).strip()should_run_preprocess = Falseif preprocess_val == 'True':    should_run_preprocess = Trueelif preprocess_val == 'auto':    import os, glob    wav_match = re.search(r"wav_dir:\s*str\s*=\s*[\"']([^\"']+)[\"']", config_content)    preprocess_dir_match = re.search(r"preprocessed_dir\s*=\s*[\"']([^\"']+)[\"']", config_content)    if wav_match and preprocess_dir_match:        wav_dir = wav_match.group(1)        preprocessed_dir = preprocess_dir_match.group(1)        if not os.path.exists(preprocessed_dir):            should_run_preprocess = True        else:            wav_count = len(glob.glob(os.path.join(wav_dir, '*.wav')))            pt_count = len(glob.glob(os.path.join(preprocessed_dir, '*.pt')))            if wav_count != pt_count:                should_run_preprocess = Trueelse:    should_run_preprocess = Trueif should_run_preprocess:    print("\n⏳ Starting preprocessing (this may take several minutes)...")    !python train.py    print("\n✅ Preprocessing completed!")    print("\n" + "="*60)    print("🏋️ Starting training...")    print("="*60)    print("Training progress and audio samples will be displayed below.")    print("Press Ctrl+C to stop training early.")    !python train.pyelse:    print("\n⏳ Skipping preprocessing (preprocess=False or auto-detected as complete)")    print("\n🏋️ Starting training...")    print("="*60)    print("Training progress and audio samples will be displayed below.")    print("Press Ctrl+C to stop training early.")    !python train.pyprint("\n✅ Training completed!")print("Check the chatterbox_output/ directory for your trained model.")# Check if auto-merge is enabledis_lora_match = re.search(r'is_lora:\s*bool\s*=\s*(\w+)', config_content)is_merge_lora_match = re.search(r'is_merge_lora:\s*bool\s*=\s*(\w+)', config_content)is_lora = is_lora_match and is_lora_match.group(1) == 'True'is_merge_lora = is_merge_lora_match and is_merge_lora_match.group(1) == 'True'if is_lora and is_merge_lora:    print("\n" + "="*60)    print("🔄 Auto-merging LoRA adapter (is_merge_lora=True)...")    print("="*60)    !python merge_lora.py    print("\n✅ LoRA adapter merged successfully!")else:    print("\n" + "="*60)    print("ℹ️ Auto-merge skipped:")    if not is_lora:        print("   - LoRA training was not used (is_lora=False)")    if not is_merge_lora:        print("   - Auto-merge is disabled (is_merge_lora=False)")    print("   - You can manually merge later in OPTIONAL STEP 6")    print("="*60)

## OPTIONAL STEP 6: Merge LoRA Adapter (Manual)If you trained with LoRA and didn't enable auto-merge, use this step to manually merge the adapter.

In [ ]:
#@title 📦 OPTIONAL STEP 6: Merge LoRA Adapter (Manual)%cd /content/chatterbox-finetuningprint("Merging LoRA adapter into base model...")print("This creates a standalone .safetensors file that doesn't require LoRA adapters.")!python merge_lora.pyprint("\n✅ Merge completed!")print("Your merged model is ready for production use.")print("Check chatterbox_output/ for the merged .safetensors file.")

## OPTIONAL STEP 7: Inference (Generate Speech)Test your trained model by generating speech from text.

In [ ]:
#@title 🗣️ OPTIONAL STEP 7: Inference - Generate Speech%cd /content/chatterbox-finetuning# First, let's check what models are availableprint("Checking available models...")!ls -la chatterbox_output/print("\n" + "="*60)print("To generate speech, edit inference.py with your desired text:")print("  TEXT_TO_SAY = \"Your text here\"")print("  AUDIO_PROMPT = \"./speaker_reference/reference.wav\"")print("="*60)# Run inferenceprint("\nRunning inference...")!python inference.pyprint("\n✅ Speech generated!")print("Output saved as output.wav")# Play the generated audiofrom IPython.display import Audio, displayprint("\nPlaying generated audio:")display(Audio('./output.wav'))